In [0]:
from pyspark.sql import functions as F

# Leer tabla Bronze
df_bronze = spark.table(
    "world_bank.costa_rica.bronze_economic_indicators"
)

display(df_bronze)

In [0]:
# Definir la estructura del JSON
# Los archivos de World Bank contienen dos elementos principales: los metadatos de la consulta y una lista de registros económicos.
from pyspark.sql.types import (
    ArrayType, StructType, StructField,
    StringType, DoubleType
)

# Estructura de cada registro económico
registro_schema = StructType([
    StructField("countryiso3code", StringType()),
    StructField("date", StringType()),
    StructField("value", DoubleType()),
    StructField("indicator", StructType([
        StructField("id", StringType()),
        StructField("value", StringType())
    ])),
    StructField("country", StructType([
        StructField("id", StringType()),
        StructField("value", StringType())
    ]))
])

# Estructura completa: [metadatos, registros]
json_schema = ArrayType(
    StringType()
)

In [0]:
# Obtener la lista de registros
df_json = df_bronze.withColumn(
    "records_json",
    F.get_json_object(
        F.col("raw_json"),
        "$[1]"
    )
)

# Convertir los registros a un array estructurado
df_json = df_json.withColumn(
    "records",
    F.from_json(
        F.col("records_json"),
        ArrayType(registro_schema)
    )
)

# Convertir cada elemento del array en una fila
df_records = df_json.select(
    F.explode("records").alias("record")
)

display(df_records.limit(5))

In [0]:
# Seleccionar y transformar columnas
df_silver = df_records.select(
    F.col("record.countryiso3code").alias("country_code"),
    F.col("record.country.value").alias("country"),
    F.col("record.date").cast("int").alias("year"),
    F.col("record.indicator.id").alias("indicator_code"),
    F.col("record.indicator.value").alias("indicator_name"),
    F.col("record.value").alias("value")
)

display(df_silver.limit(5))

In [0]:
# Eliminaremos registros duplicados y aquellos que no tengan un valor económico disponible.
df_silver = (
    df_silver
    # Eliminar duplicados por indicador y año
    .dropDuplicates([
        "country_code",
        "year",
        "indicator_code"
    ])

    # Eliminar registros sin valor
    .filter(F.col("value").isNotNull())

    # Ordenar los registros
    .orderBy("year", "indicator_code")
)

display(df_silver.limit(5))

In [0]:
# Guardar la tabla Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "world_bank.costa_rica.silver_economic_indicators"
    )

print("Tabla Silver creada correctamente")